In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

In [2]:
path = "/kaggle/input/plantvillage-dataset/color"
BATCH_SIZE = 32

In [3]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize 224x224
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),                         # Converts to [0,1]
])

In [4]:
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),                         # Converts to [0,1]
])

In [5]:
full_dataset = datasets.ImageFolder(root=path)  # temporary, will split next)

class_names = full_dataset.classes
num_classes = len(class_names)

print("Number of classes:", num_classes)

Number of classes: 38


In [6]:
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size]
)

In [7]:
train_dataset.dataset.transform = train_transforms
val_dataset.dataset.transform = val_transforms

In [8]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

In [9]:
images, labels = next(iter(train_loader))
print("Image batch shape:", images.shape)   # [B, 3, 224, 224]
print("Label batch shape:", labels.shape)

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [11]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2, 2)
        
        # Fully connected layers
        self.fc1 = nn.Linear(64 * 56 * 56, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # [B, 32, 112, 112]
        x = self.pool(F.relu(self.conv2(x)))   # [B, 64, 56, 56]
        
        x = x.view(x.size(0), -1)               # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [12]:
num_classes = len(class_names)

model = SimpleCNN(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [13]:
def train_model(model, train_loader, val_loader, epochs):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        train_acc = 100 * correct / total
        
        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_acc = 100 * val_correct / val_total
        
        print(f"Epoch [{epoch+1}/{epochs}] "
              f"Loss: {running_loss:.4f} "
              f"Train Acc: {train_acc:.2f}% "
              f"Val Acc: {val_acc:.2f}%")

In [14]:
train_model(
    model,
    train_loader,
    val_loader,
    epochs=10
)

Epoch [1/10] Loss: 1368.5541 Train Acc: 70.79% Val Acc: 84.21%
Epoch [2/10] Loss: 492.6680 Train Acc: 88.39% Val Acc: 83.33%
Epoch [3/10] Loss: 258.5233 Train Acc: 93.82% Val Acc: 88.32%
Epoch [4/10] Loss: 160.5765 Train Acc: 96.09% Val Acc: 84.99%
Epoch [5/10] Loss: 123.2943 Train Acc: 97.03% Val Acc: 86.49%
Epoch [6/10] Loss: 104.1380 Train Acc: 97.57% Val Acc: 87.80%
Epoch [7/10] Loss: 85.9956 Train Acc: 97.95% Val Acc: 87.03%
Epoch [8/10] Loss: 68.8413 Train Acc: 98.36% Val Acc: 88.10%
Epoch [9/10] Loss: 62.0162 Train Acc: 98.55% Val Acc: 88.41%
Epoch [10/10] Loss: 60.8543 Train Acc: 98.61% Val Acc: 88.27%
